# Задание 8. Компьютерное зрение с глубоким обучением: GAN

## Вариант: базовый

Требования (по readme.md):

1) Расписать код (запускать код не обязательно).  
2) Ниже добавить текстовую ячейку с описанием того, как работает код.  
3) Сделать 2 диаграммы моделей GAN (Generator и Discriminator) — **разместить над кодом модели**.


### Задание 1. Импорты и настройки

- Используем PyTorch + torchvision.
- Датасет: MNIST (скачивается автоматически).
- Для дискриминатора используем `BCEWithLogitsLoss` (логиты без `sigmoid`).


In [ ]:
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.utils as vutils

import matplotlib.pyplot as plt


In [ ]:
RANDOM_SEED = 42
BATCH_SIZE = 128
NUM_WORKERS = 2

IMAGE_SIZE = 28
CHANNELS = 1  # MNIST: grayscale

LATENT_DIM = 100
G_FEATURES = 64
D_FEATURES = 64

LR = 2e-4
BETAS = (0.5, 0.999)
EPOCHS = 20

# Воспроизводимость
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


### Задание 2. Данные (MNIST)

Нормировка делается в диапазон **[-1, 1]**, чтобы выход генератора с `tanh` был согласован по масштабу.


In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),  # -> [-1, 1]
    ]
)

train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

x0, _ = next(iter(train_loader))
print("batch:", x0.shape, "min/max:", float(x0.min()), float(x0.max()))


In [ ]:
# Визуализация нескольких реальных изображений (для проверки данных)

real_batch = x0[:64]

grid = vutils.make_grid(
    real_batch,
    nrow=8,
    normalize=True,
    value_range=(-1, 1),
)

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("MNIST: примеры реальных изображений")
plt.imshow(np.transpose(grid.cpu(), (1, 2, 0)))
plt.show()


### Задание 3. Диаграммы моделей GAN

**Generator (G):** `z -> image`  

```
z: [B, 100]
  -> Linear(100 -> 256*7*7) + BatchNorm + ReLU
  -> reshape: [B, 256, 7, 7]
  -> ConvTranspose2d(256 -> 128, k=4, s=2, p=1) + BN + ReLU   => [B, 128, 14, 14]
  -> ConvTranspose2d(128 -> 64,  k=4, s=2, p=1) + BN + ReLU   => [B,  64, 28, 28]
  -> Conv2d(64 -> 1, k=3, s=1, p=1) + Tanh                    => [B,   1, 28, 28]
```

**Discriminator (D):** `image -> logit` (без `sigmoid`, т.к. используем `BCEWithLogitsLoss`)  

```
x: [B, 1, 28, 28]
  -> Conv2d(1 -> 64,  k=4, s=2, p=1) + LeakyReLU              => [B,  64, 14, 14]
  -> Conv2d(64 -> 128, k=4, s=2, p=1) + BN + LeakyReLU        => [B, 128,  7,  7]
  -> Conv2d(128 -> 256, k=4, s=2, p=1) + BN + LeakyReLU       => [B, 256,  3,  3]
  -> AdaptiveAvgPool2d(1)                                     => [B, 256,  1,  1]
  -> Flatten                                                  => [B, 256]
  -> Linear(256 -> 1)                                         => [B,   1]  (logit)
```

`AdaptiveAvgPool2d(1)` добавлен, чтобы размерность перед `Linear` была стабильной и не было ошибок вида
`mat1 and mat2 shapes cannot be multiplied`.


### Задание 4. Код моделей GAN

Ниже — реализация `Generator` и `Discriminator` в соответствии с диаграммами.


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim: int, channels: int, features: int) -> None:
        super().__init__()

        # z -> [B, 256*7*7]
        self.proj = nn.Sequential(
            nn.Linear(latent_dim, features * 4 * 7 * 7),
            nn.BatchNorm1d(features * 4 * 7 * 7),
            nn.ReLU(inplace=True),
        )

        # [B, 256, 7, 7] -> [B, 1, 28, 28]
        self.net = nn.Sequential(
            nn.ConvTranspose2d(
                features * 4,
                features * 2,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(features * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(
                features * 2,
                features,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
            nn.Conv2d(features, channels, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.proj(z)
        x = x.view(z.size(0), -1, 7, 7)
        img = self.net(x)
        return img


class Discriminator(nn.Module):
    def __init__(self, channels: int, features: int) -> None:
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(channels, features, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(
                features,
                features * 2,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(
                features * 2,
                features * 4,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(features * 4, 1),  # logit
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.net(x)
        logit = self.head(h)
        return logit


In [ ]:
G = Generator(LATENT_DIM, CHANNELS, G_FEATURES).to(device)
D = Discriminator(CHANNELS, D_FEATURES).to(device)

print("G params:", sum(p.numel() for p in G.parameters()))
print("D params:", sum(p.numel() for p in D.parameters()))


### Задание 5. Loss и оптимизаторы

- `BCEWithLogitsLoss` ожидает **логиты** (без `sigmoid`).
- Для DCGAN-подобных сетей часто используют Adam с `betas=(0.5, 0.999)`.


In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=BETAS)
optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=BETAS)

fixed_noise = torch.randn(64, LATENT_DIM, device=device)


### Задание 6. Цикл обучения (опционально)

In [ ]:
losses_D = []
losses_G = []

for epoch in range(1, EPOCHS + 1):
    for i, (real, _) in enumerate(train_loader):
        real = real.to(device, non_blocking=True)
        bsz = real.size(0)

        # Метки
        y_real = torch.ones((bsz, 1), device=device)
        y_fake = torch.zeros((bsz, 1), device=device)

        # Обучение дискриминатора
        D.train()
        G.eval()

        optimizer_D.zero_grad(set_to_none=True)

        logits_real = D(real)
        loss_real = criterion(logits_real, y_real)

        noise = torch.randn(bsz, LATENT_DIM, device=device)
        fake = G(noise).detach()
        logits_fake = D(fake)
        loss_fake = criterion(logits_fake, y_fake)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D.step()

        # Обучение генератора
        D.eval()
        G.train()

        optimizer_G.zero_grad(set_to_none=True)

        noise = torch.randn(bsz, LATENT_DIM, device=device)
        fake = G(noise)
        logits = D(fake)

        # Лайфъак дискриминатор
        loss_G = criterion(logits, y_real)
        loss_G.backward()
        optimizer_G.step()

        losses_D.append(float(loss_D.item()))
        losses_G.append(float(loss_G.item()))

        if i % 200 == 0:
            print(
                f"epoch {epoch:02d}/{EPOCHS} | step {i:04d}/{len(train_loader)} "
                f"| lossD={loss_D.item():.4f} lossG={loss_G.item():.4f}"
            )

    with torch.no_grad():
        G.eval()
        fake = G(fixed_noise).cpu()

    grid = vutils.make_grid(
        fake,
        nrow=8,
        normalize=True,
        value_range=(-1, 1),
    )

    plt.figure(figsize=(8, 8))
    plt.axis("off")
    plt.title(f"Fake samples (epoch {epoch})")
    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.show()


In [ ]:
if len(losses_D) > 0 and len(losses_G) > 0:
    plt.figure(figsize=(10, 4))
    plt.plot(losses_D, label="loss_D")
    plt.plot(losses_G, label="loss_G")
    plt.legend()
    plt.grid(True)
    plt.title("GAN losses")
    plt.show()


### Пояснение: как работает код (GAN)

**1) Данные.** Берём MNIST и нормируем изображения в диапазон **[-1, 1]**. Это согласовано с выходом генератора (`tanh`), который тоже выдаёт значения в [-1, 1].

**2) Генератор (G).** Получает на вход шум `z` (`LATENT_DIM=100`) и превращает его в изображение:
- `Linear + BN + ReLU` строит заготовку признаков размера `256×7×7`;
- `ConvTranspose2d` увеличивает разрешение до 28×28;
- `Conv2d + tanh` даёт итоговый кадр 1×28×28.

**3) Дискриминатор (D).** Получает изображение и выдаёт **логит** (одно число):
- несколько `Conv2d` с downsample выделяют признаки;
- `AdaptiveAvgPool2d(1)` делает пространственный размер 1×1 независимо от деталей свёрток;
- `Linear` выдаёт логит, который подаётся в `BCEWithLogitsLoss`.

**4) Обучение.**  
- Шаг D: минимизируем ошибку распознавания `real` как 1 и `fake` как 0. Для `fake` используем `detach()`, чтобы не считать градиенты для G.  
- Шаг G: генерируем `fake` и хотим, чтобы D выдавал для него 1 (генератор “обманывает” дискриминатор).
